In [1]:
!pip install torch transformers

In [2]:
import json
import os
import torch
CLASSES = {
    "Adware": 0,
    "Backdoor": 1,
    "Botnet": 2,
    "CGI": 3,
    "Code-execution": 4,
    "DDos": 5,
    "Dir-Traversal": 6,
    "Dos": 7,
    "Info-Disclosure": 8,
    "Injection": 9,
    "Other": 10,
    "Overflow": 11,
    "Ransomware": 12,
    "Remote-file-Inclusion": 13,
    "Scanner": 14,
    "Spyware": 15,
    "Trojan": 16,
    "Virus": 17,
    "Webshell": 18,
    "Worm": 19,
    "XSS": 20
}
INV_CLASSES = {v: k for k, v in CLASSES.items()}
CONCEPTS= ["ip", "injection"]
CLASSES_TO_EXAMINE = ["Adware", "Scanner", "Spyware", "Trojan", "XSS", "Remote-file-Inclusion", "Overflow", "Injection", "Info-Disclosure", "Dir-Traversal", "Code-execution", "CGI", "Ransomware", "Botnet", "Backdoor"]
MODEL_NAME = "./codebert-base-mlm"


from transformers import AutoModelForSequenceClassification, AutoTokenizer

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, output_hidden_states=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [3]:
MAIN_DATASETS = {}
def load_dataset(name):
    with open(f"./data/packet_inspection/{name}.jsonl", "r") as f:
        return [json.loads(line) for line in f]
    

MAIN_DATASETS["complete"] = load_dataset("packets_dataset")
MAIN_DATASETS["ip"] = load_dataset("ip_dataset")
MAIN_DATASETS["injection"] = load_dataset("injection_dataset")


CONCEPT_TO_DATASET = {
    "ip": MAIN_DATASETS["ip"],
    "injection": MAIN_DATASETS["injection"],
}

In [17]:
import random

data = random.sample(MAIN_DATASETS["complete"],1)

text = [s["text"] for s in data]
print(text)
inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

with torch.no_grad():
    outputs = model(**inputs)

hidden_states = outputs.hidden_states

print(len(hidden_states), hidden_states[0].shape)
print(hidden_states[0])

["10.0.0.6\nGET /vuln.php?assert=system('ls') HTTP/1.1\nHost: files.com\nUser-Agent: curl/7.80.0\nAccept: text/html\nAccept-Encoding: gzip, deflate\nAccept-Charset: UTF-8\nAccept-Language: en-US\nConnection: close\n"]
13 torch.Size([1, 87, 768])
tensor([[[ 0.1532, -0.0733, -0.0070,  ...,  0.0055,  0.1258,  0.0745],
         [-0.1335,  0.3062,  0.3961,  ...,  0.0116,  0.0213,  0.4741],
         [ 0.3971,  0.0206,  0.0192,  ..., -0.9080, -0.1878,  0.2998],
         ...,
         [ 0.2378, -0.1530, -0.1386,  ...,  0.4655, -0.4601, -0.0043],
         [ 0.1335, -0.0772, -0.1529,  ...,  0.4598,  0.0362,  0.1027],
         [-0.0175, -0.1342,  0.1402,  ...,  0.2782,  0.3829, -0.0794]]])
13 torch.Size([1, 87, 768])
tensor([[[ 0.1532, -0.0733, -0.0070,  ...,  0.0055,  0.1258,  0.0745],
         [-0.1335,  0.3062,  0.3961,  ...,  0.0116,  0.0213,  0.4741],
         [ 0.3971,  0.0206,  0.0192,  ..., -0.9080, -0.1878,  0.2998],
         ...,
         [ 0.2378, -0.1530, -0.1386,  ...,  0.4655, -0.46

In [16]:
print(model)

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
         

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
from sklearn.linear_model import SGDClassifier
import random
import string

In [ ]:
class TCAV:
    """ Class for concept activation vectors for PyTorch models.

    Attributes:
        model: a roBerta LLM loaded through Huggingface API
        tokenizer: a tokenizer for roBerta
        cavs: dict mapping concept names to their activation vectors
        sensitivities: dict mapping concept names to their sensitivities
        y_labels: list that will contain labels of the values used to calculate sensitivities
        bottleneck: list of bottleneck layers to analyze
        model_activations: dict storing activations and gradients for hooks
        fix_length: optional fixed sequence length for tokenization
    """

    def __init__(self, model=None, tokenizer=None, fix_length=None):
        self.model = model
        self.tokenizer = tokenizer
        self.cavs = {}  # dict: concept_name -> cav
        self.sensitivities = {}  # dict: concept_name -> sensitivities
        self.y_labels = None # list of testing labels concept_name -> y_labels
        self.current_concept = None
        self.bottleneck = [] # list of bottleneck layers to analyze
        self.model_activations = {}
        self.fix_length = fix_length
    
    
    def backward_hook_fn(self, name:str):
        if name not in self.model_activations.keys():
            self.model_activations["backward_"+name] = []
            
        def fn(module, grad_input, grad_output):
            # print("leaf",grad_output[0].is_leaf)
            
            self.model_activations["backward_"+name] = grad_output[0]
            # print("extracted grads", grad_output[0].shape)
            return
        
        return fn
    
    def split_model(self, bottleneck):
        """ Set the hook at the bottleneck layer """
        if bottleneck < 0 or bottleneck >= len(self.model.roberta.encoder.layer):
            raise ValueError("Invalid layer for sampling")
        
        # layers = list(self.model.children())
        # print(layers)
        self.bottleneck.append(str(bottleneck))   
        
        self.model.roberta.encoder.layer[bottleneck].register_full_backward_hook(self.backward_hook_fn(str(bottleneck)))
        return
    
    def set_concept(self, concept):
        """ Set the concept name """
        self.current_concept = concept

    def set_model(self, model):
        """ Set the model """
        self.model = model
        return

    def set_tokenizer(self, tokenizer):
        """ Set the tokenizer """
        self.tokenizer = tokenizer
        return

    def _create_counterexamples(self, x_concept):
        """ Creates random counterexamples to a series of concept inputs """
        n = len(x_concept)
    
        counterexamples = []
        for i in range(n):
            l = len(x_concept[i])
            counterexamples.append(''.join(random.choices(string.printable, k=l)))
        return counterexamples

    def _tokenize(self, inputs):
        """ Tokenize the inputs (if tokenizer is provided) """
        # print(len(inputs))
        if self.tokenizer is not None:
            if self.fix_length:
                x = self.tokenizer(inputs, return_tensors="pt", padding="max_length", max_length=self.fix_length, truncation=True)
            else:
                x = self.tokenizer(inputs, return_tensors="pt", padding="longest", truncation=True)
                self.fix_length = x["input_ids"].shape[1]
            # print("tokenizer", x["input_ids"].shape)
            return x
        return inputs

    def train_cav(self, x_concept):
        """ Train and extract the Concept Activation Vector """
        # print(f"concept: {len(x_concept)}")
        counterexamples = self._create_counterexamples(x_concept)
        tmp = x_concept + counterexamples
        # print(f"complete: {len(tmp)}")
        x_train_concept = self._tokenize(tmp)
        # print(x_train_concept,  x_train_concept["input_ids"].shape)
        y_train_concept = torch.cat((torch.ones(len(x_concept)), torch.zeros(len(counterexamples))))
        
        # print("calculating cavs")
        # Obtain activations of concept and counterexamples
        with torch.no_grad():
            outputs = self.model(**x_train_concept)
            # print("attentions dimensions:", len(self.model_activations["forward_"+self.bottleneck][0].shape))
            hidden_states = outputs.hidden_states
            concept_activations = {}
            for bottleneck in self.bottleneck:
                # print(hidden_states[int(bottleneck)].shape)
                if(len(hidden_states[int(bottleneck)].shape)>2):
                    # Flatten the activations if they are not linear: (n, m, z) to (n, m*z)
                    concept_activations[bottleneck] = hidden_states[int(bottleneck)].reshape(hidden_states[int(bottleneck)].shape[0],-1)
                else:
                    # If activations are linear, no flattening need: (n, m)
                    concept_activations[bottleneck] = hidden_states[int(bottleneck)]
                # print("concept activations shape", concept_activations.shape)
        # print(concept_activations.shape)

        # Iterate over all bottlenecks
        # print("bottlenecks", self.bottleneck)
        self.cavs[self.current_concept] = {}

        for b in concept_activations.keys():
            # Train linear classifier
            lm = SGDClassifier(loss="perceptron", eta0=1, learning_rate="constant", penalty=None)

            # print(f"{b}: {y_train_concept}")
            
            lm.fit(concept_activations[b].detach().numpy(), y_train_concept.numpy())
            cav = -lm.coef_.T
            
            self.cavs[self.current_concept][b] = cav
            # print("cav", len(cav))
            
            # reset gradients
            self.model_activations["backward_"+b] = []
        
        return
        
    def calculate_sensitivity(self, x_train, y_train, device="cpu"):
        """
        This function calculates the TCAV sensitivity scores for a given set of inputs and labels.
        It computes the gradient of the loss with respect to the activations at the bottleneck layer,
        projects these gradients onto the concept activation vector (CAV), and measures how sensitive
        the model's predictions are to the concept for each class. The results are stored for later analysis.
        """
        
        # print("calculating sensitivity")
        x_train = self._tokenize(x_train)
        # print(x_train)
        
        # Calculate the output and obtain activations
        x_train = x_train.to(device)
        output = self.model(**x_train)
        hidden_states = output.hidden_states
        
        #activations = self.model_activations["forward_"+self.bottleneck][0].reshape(self.model_activations["forward_"+self.bottleneck][0].shape[0],-1) # Prendi l'ultima attivazione del bottleneck
        # print("output logits", output.logits.shape)
        # print("activation shape", activations.shape)
        
        # Control the format
        if isinstance(y_train, list):
            y_train = np.array(y_train)
            # print(y_train)
        if not isinstance(y_train, torch.Tensor):
            y_train = torch.from_numpy(y_train)
        y_labels = y_train.view(-1).to(device)
        # print(y_labels)

        # Define and compute the loss
        loss = F.cross_entropy(output.logits, y_labels)
        # print("loss", loss)
        # print("activations",activations.is_leaf, activations)
        # print("activations requires grad", activations.requires_grad)

        # Calculate the gradient
        loss.backward()
        
        grads = {}
        for bottleneck in self.bottleneck:
            grads[bottleneck] = self.model_activations["backward_"+bottleneck]
    
        # grads = torch.autograd.grad(loss, activations, allow_unused=True)
        # print("grads", grads)
        # concatenate grads
        grads = {k: v.reshape(v.shape[0], -1) for k, v in grads.items()}

        # Scalar product
        cavs = self.cavs
        
        # print("shapes", cavs.shape, grads.shape)
        sensitivities = {}
        for concept in cavs.keys():
            sensitivities[concept] = {}
            cav_tensor = cavs[concept]
            for b in self.bottleneck:
            # print("cav_tensor", cav_tensor.shape)
                sensitivities[concept][b] = (np.dot(grads[b], cav_tensor[b]))
        
        
        # print("sensitivity", sensitivity)

        # Saving sensitivity
        del grads, cavs
        self.sensitivities = sensitivities
        self.y_labels = y_train.detach().cpu().numpy().reshape(-1)
        for bottleneck in self.bottleneck:
            self.model_activations["backward_"+bottleneck] = []

        return
        
    def print_all_sensitivities(self, id_to_labels):
        for concept, sensitivities_dict in self.sensitivities.items():
            print(f"Sensitivities for concept '{concept}':")
            for bottleneck, sensitivity in sensitivities_dict.items():
                print(f"  Bottleneck {bottleneck}:")
                num_labels = len(np.unique(self.y_labels))
                for label_idx in range(num_labels):
                    idxs = np.where(self.y_labels == label_idx)[0]
                    value = np.sum(sensitivity[idxs] > 0) / idxs.shape[0]
                    print(f"    Class {id_to_labels[label_idx]}: {value:.2f}")
                print("-" * 5)

    def get_sensitivity_results(self, id_to_labels):
        """
        Returns a list of dicts with the sensitivity results for all concepts, bottlenecks and classes.
        Format:
        [
            {"concept": "...", "bottleneck": "...", "class": "...", "sensitivity": 0.xx},
            ...
        ]
        """
        results = []
        for concept, sensitivities_dict in self.sensitivities.items():
            for bottleneck, sensitivity in sensitivities_dict.items():
                num_labels = len(np.unique(self.y_labels))
                for label_idx in range(num_labels):
                    idxs = np.where(self.y_labels == label_idx)[0]
                    value = np.sum(sensitivity[idxs] > 0) / idxs.shape[0]
                    results.append({
                        "concept": concept,
                        "bottleneck": bottleneck,
                        "class": id_to_labels[label_idx],
                        "sensitivity": value
                    })
        return results

In [ ]:
class TCAV_Avg:
    """ Class for concept activation vectors for PyTorch models.

    Attributes:
        model: a roBerta LLM loaded through Huggingface API
        tokenizer: a tokenizer for roBerta
        cavs: dict mapping concept names to their activation vectors
        sensitivities: dict mapping concept names to their sensitivities
        y_labels:  list that will contain labels of the values used to calculate sensitivities
        bottleneck: list of bottleneck layers to analyze
        model_activations: dict storing activations and gradients for hooks
        fix_length: optional fixed sequence length for tokenization
    """

    def __init__(self, model=None, tokenizer=None, fix_length=512):
        self.model = model
        self.tokenizer = tokenizer
        self.cavs = {}  # dict: concept_name -> cav
        self.sensitivities = {}  # dict: concept_name -> sensitivities
        self.y_labels = None  # list of testing labels -> y_labels
        self.current_concept = None
        self.bottleneck = [] # list of bottleneck layers to analyze
        self.model_activations = {}
        self.fix_length = fix_length
    
    
    def backward_hook_fn(self, name:str):
        if name not in self.model_activations.keys():
            self.model_activations["backward_"+name] = []
            
        def fn(module, grad_input, grad_output):
            # print("leaf",grad_output[0].is_leaf)
            
            self.model_activations["backward_"+name] = grad_output[0]
            # print("extracted grads", grad_output[0].shape)
            return
        
        return fn

    def set_concept(self, concept):
        """ Set the concept name """
        self.current_concept = concept
        
    def set_model(self, model):
        """ Set the model """
        self.model = model
        return

    def set_tokenizer(self, tokenizer):
        """ Set the tokenizer """
        self.tokenizer = tokenizer
        return

    def split_model(self, bottleneck):
        """ Set the hook at the bottleneck layer """
        if bottleneck < 0 or bottleneck >= len(self.model.roberta.encoder.layer):
            raise ValueError("Invalid layer for sampling")
        
        # layers = list(self.model.children())
        # print(layers)
        self.bottleneck.append(str(bottleneck))   
        
        # self.model.classifier.dense.register_forward_hook(self.hook_fn(str(bottleneck)))
        self.model.roberta.encoder.layer[bottleneck].register_full_backward_hook(self.backward_hook_fn(str(bottleneck)))
        return


    def _create_counterexamples(self, x_concept):
        """ Creates random counterexamples to a series of concept inputs """
        n = len(x_concept)
    
        counterexamples = []
        for i in range(n):
            l = len(x_concept[i])
            counterexamples.append(''.join(random.choices(string.printable, k=l)))
        return counterexamples

    def _tokenize(self, inputs):
        """ Tokenize the inputs (if tokenizer is provided) """
        # print(len(inputs))
        if self.tokenizer is not None:
            x = self.tokenizer(inputs, return_tensors="pt", padding="max_length", max_length=self.fix_length, truncation=True)
            # print("tokenizer", x["input_ids"].shape)
            return x
        return inputs

    def train_cav(self, x_concept):
        """ Train and extract the Concept Activation Vector """
        
        counterexamples = self._create_counterexamples(x_concept)
        tmp = x_concept + counterexamples
        x_train_concept = self._tokenize(tmp)
        y_train_concept = torch.cat((torch.ones(len(x_concept)), torch.zeros(len(counterexamples))))
        
        # print("calculating cavs")
        # Obtain activations of concept and counterexamples
        with torch.no_grad():
            outputs = self.model(**x_train_concept)
            
            concept_activations = {}
            for bottleneck in self.bottleneck:
                hidden_states = outputs.hidden_states
                # print(hidden_states[int(bottleneck)].shape)
                if(len(hidden_states[int(bottleneck)].shape)>2):
                    # Instead of flattening (n, m, z) to (n, m*z), take the mean over m to get (n, z)
                    # print("attentions dimensions:", len(self.model_activations["forward_"+bottleneck][0].shape))
                    activations = hidden_states[int(bottleneck)]  # (batch, seq_len, hidden)
                    attention_mask = x_train_concept["attention_mask"]  # (batch, seq_len)
                    # print("attention mask", attention_mask.shape, attention_mask)
                    mask = attention_mask.unsqueeze(-1).expand(activations.size())  # (batch, seq_len, hidden)
                    activations_masked = activations * mask  # maschera i padding
                    lengths = attention_mask.sum(dim=1).unsqueeze(-1)  # (batch, 1)
                    # Evita divisione per zero
                    lengths = lengths.clamp(min=1)
                    concept_activations[bottleneck] = activations_masked.sum(dim=1) / lengths
                else:
                    concept_activations[bottleneck] = self.model_activations["forward_"+bottleneck][0]
                # print("concept activations shape", concept_activations.shape)
        # print(concept_activations.shape)
        
        # Iterate over all bottlenecks
        # print("bottlenecks", self.bottleneck)
        self.cavs[self.current_concept] = {}

        for b in concept_activations.keys():
            # Train linear classifier
            lm = SGDClassifier(loss="perceptron", eta0=1, learning_rate="constant", penalty=None)
            lm.fit(concept_activations[b].detach().numpy(), y_train_concept.numpy())
            cav = -lm.coef_.T
            
            self.cavs[self.current_concept][b] = cav
            # print("cav", len(cav))
                      
            self.model_activations["backward_"+b] = []
            
        del tmp
        del x_train_concept
        del y_train_concept
        del concept_activations
        return
        
    def calculate_sensitivity(self, x_train, y_train, device="cpu"):
        """
        This function calculates the TCAV sensitivity scores for a given set of inputs and labels.
        It computes the gradient of the loss with respect to the activations at the bottleneck layer,
        projects these gradients onto the concept activation vector (CAV), and measures how sensitive
        the model's predictions are to the concept for each class. The results are stored for later analysis.
        """
        
        # print("calculating sensitivity")
        x_train = self._tokenize(x_train)
        # print(x_train)
        
        # Calculate the output and obtain activations
        x_train = x_train.to(device)
        output = self.model(**x_train)
        hidden_states = output.hidden_states
        
        # Take the mean over the sequence dimension (m) to get (n, z)
        # activations = self.model_activations["forward_"+self.bottleneck][0].mean(dim=1)
        # print("output logits", output.logits.shape)
        # print("activation shape", activations.shape)
        
        # Control the format
        if isinstance(y_train, list):
            y_train = np.array(y_train)
            # print(y_train)
        if not isinstance(y_train, torch.Tensor):
            y_train = torch.from_numpy(y_train)
        y_labels = y_train.view(-1).to(device)
        # print(y_labels)

        # Define and compute the loss
        loss = F.cross_entropy(output.logits, y_labels)
        # print("loss", loss)
        # print("activations",activations.is_leaf, activations)
        # print("activations requires grad", activations.requires_grad)

        # Calculate the gradient
        loss.backward()
        
        grads = {}
        for bottleneck in self.bottleneck:
            grads[bottleneck] = self.model_activations["backward_"+bottleneck]
            if grads[bottleneck].dim() > 2:
                attention_mask = x_train["attention_mask"]  # (batch, seq_len)
                mask = attention_mask.unsqueeze(-1).expand(grads[bottleneck].size())  # (batch, seq_len, hidden)
                grads_masked = grads[bottleneck] * mask  # maschera i padding
                lengths = attention_mask.sum(dim=1).unsqueeze(-1)  # (batch, 1)
                lengths = lengths.clamp(min=1)
                grads[bottleneck] = grads_masked.sum(dim=1) / lengths
    
        # grads = torch.autograd.grad(loss, activations, allow_unused=True)
        # print("grads", grads)
        # concatenate grads
        # grads = {k: v.reshape(v.shape[0], -1) for k, v in grads.items()}

        # Scalar product
        cavs = self.cavs
        
        # print("shapes", cavs.shape, grads.shape)
        sensitivities = {}
        for concept in cavs.keys():
            sensitivities[concept] = {}
            cav_tensor = cavs[concept]
            for b in self.bottleneck:
            # print("cav_tensor", cav_tensor.shape)
                sensitivities[concept][b] = (np.dot(grads[b], cav_tensor[b]))
        
        
        # print("sensitivity", sensitivity)

        # Saving sensitivity
        self.sensitivities = sensitivities
        self.y_labels = y_train.detach().cpu().numpy().reshape(-1)
        
        for bottleneck in self.bottleneck:
            self.model_activations["backward_"+bottleneck] = []
        del grads
        del cavs
        del x_train
        del output

        return
        
    def print_all_sensitivities(self, id_to_labels):
        for concept, sensitivities_dict in self.sensitivities.items():
            print(f"Sensitivities for concept '{concept}':")
            for bottleneck, sensitivity in sensitivities_dict.items():
                print(f"  Bottleneck {bottleneck}:")
                num_labels = len(np.unique(self.y_labels))
                for label_idx in range(num_labels):
                    idxs = np.where(self.y_labels == label_idx)[0]
                    value = np.sum(sensitivity[idxs] > 0) / idxs.shape[0]
                    print(f"    Class {id_to_labels[label_idx]}: {value:.2f}")
                print("-" * 5)

    def get_sensitivity_results(self, id_to_labels):
        """
        Returns a list of dicts with the sensitivity results for all concepts, bottlenecks and classes.

        Args:
            id_to_labels (dict): Mapping from class indices to class names.

        Returns:
        [
            {"concept": "...", "bottleneck": "...", "class": "...", "sensitivity": 0.xx},
            ...
        ]
        """
        results = []
        for concept, sensitivities_dict in self.sensitivities.items():
            for bottleneck, sensitivity in sensitivities_dict.items():
                num_labels = len(np.unique(self.y_labels))
                for label_idx in range(num_labels):
                    idxs = np.where(self.y_labels == label_idx)[0]
                    value = np.sum(sensitivity[idxs] > 0) / idxs.shape[0]
                    results.append({
                        "concept": concept,
                        "bottleneck": bottleneck,
                        "class": id_to_labels[label_idx],
                        "sensitivity": value
                    })
        return results

In [ ]:
model2 = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, output_hidden_states=True)
model3 = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, output_hidden_states=True)

In [ ]:
from datetime import datetime
import gc
import time
import sys
# Ensure the results directory exists
if not os.path.exists("./results"):
    os.makedirs("./results")
if not os.path.exists("./results/packet_inspection"):
    os.makedirs("./results/packet_inspection")

f_auto = open("./results/packet_inspection/tcav_auto.txt", "a")
f_auto.write(datetime.now().isoformat() + "\n")
f_fixed = open("./results/packet_inspection/tcav_fixed.txt", "a")
f_fixed.write(datetime.now().isoformat() + "\n")
f_avg = open("./results/packet_inspection/tcav_avg.txt", "a")
f_avg.write(datetime.now().isoformat() + "\n")
f_auto.close()
f_fixed.close()
f_avg.close()
import json
import random

print(f"------------------- Executing Run -------------------")
# Initialize TCAV instances for different configurations
tcav_auto = TCAV(model=model, tokenizer=tokenizer)
tcav_fixed = TCAV(model=model2, tokenizer=tokenizer, fix_length=512)
tcav_avg = TCAV_Avg(model=model3, tokenizer=tokenizer)

# Random extract samples from main datasets
samples = {}

samples["random"] = random.sample(MAIN_DATASETS["complete"], len(MAIN_DATASETS["complete"]))
samples["test"] = random.sample(MAIN_DATASETS["complete"], len(MAIN_DATASETS["complete"]))

# Extract samples for each concept
for concept in CONCEPTS:
    samples[concept] = random.sample(CONCEPT_TO_DATASET[concept], len(CONCEPT_TO_DATASET[concept]))
    # print(f"concept lenght: {len(samples[concept])}")

for dataset_name, dataset in MAIN_DATASETS.items():
    samples[dataset_name] = random.sample(dataset, len(dataset))

# Set the bottleneck layer
for layer in range (12):
    tcav_auto.split_model(layer)
    tcav_fixed.split_model(layer)
    tcav_avg.split_model(layer)

# -------------------------------- Training TCAVs -------------------------------- 
# Train CAVs for random baseline
tcav_auto.set_concept("random")
tcav_fixed.set_concept("random")
tcav_avg.set_concept("random")

# Train CAVs for auto and fixed length
tcav_auto.train_cav([s["text"] for s in samples["random"]])
tcav_fixed.train_cav([s["text"] for s in samples["random"]])

# Train CAV for average pooling
tcav_avg.train_cav([s["text"] for s in samples["random"]])

# Train CAVs for each concept
for concept in CONCEPTS:
    tcav_auto.set_concept(concept)
    tcav_fixed.set_concept(concept)
    tcav_avg.set_concept(concept)
    
    # Train CAVs for auto and fixed length
    print(f"Training CAV for concept: {concept}")
    print("Auto TCAV training...")
    tcav_auto.train_cav([s["text"] for s in samples[concept]])
    print("Fixed TCAV training...")
    tcav_fixed.train_cav([s["text"] for s in samples[concept]])

    # Train CAV for average pooling
    print("Avg TCAV training...")
    tcav_avg.train_cav([s["text"] for s in samples[concept]])

# print("finished training")
# -------------------------------- Calculating Sensitivities --------------------------------

print("--------- Calculating sensitivities --------")
tcav_auto.calculate_sensitivity([s["text"] for s in samples["test"]], [CLASSES[s["class"]] for s in samples["test"]])
tcav_fixed.calculate_sensitivity([s["text"] for s in samples["test"]], [CLASSES[s["class"]] for s in samples["test"]])
tcav_avg.calculate_sensitivity([s["text"] for s in samples["test"]], [CLASSES[s["class"]] for s in samples["test"]])

f_auto = open("./results/packet_inspection/tcav_auto.txt", "a")
f_fixed = open("./results/packet_inspection/tcav_fixed.txt", "a")
f_avg = open("./results/packet_inspection/tcav_avg.txt", "a")

f_auto.write(json.dumps(tcav_auto.get_sensitivity_results(INV_CLASSES), indent=4)+ "\n")
f_fixed.write(json.dumps(tcav_fixed.get_sensitivity_results(INV_CLASSES), indent=4)+ "\n")
f_avg.write(json.dumps(tcav_avg.get_sensitivity_results(INV_CLASSES), indent=4)+ "\n")

f_auto.close()
f_fixed.close()
f_avg.close()

# Print sensitivities
print("Auto TCAV Sensitivities:")
tcav_auto.print_all_sensitivities(INV_CLASSES)

print("Fixed TCAV Sensitivities:")
tcav_fixed.print_all_sensitivities(INV_CLASSES)

print("Avg TCAV Sensitivities:")
tcav_avg.print_all_sensitivities(INV_CLASSES)